# はじめてのDatabricks

このノートブックではDatabricksの基本的な使い方をご説明します。

[はじめてのDatabricks](https://qiita.com/taka_yayoi/items/8dc72d083edb879a5e5d)

## Databricksの使い方

![how_to_use.png](./how_to_use.png "how_to_use.png")

## 画面の説明

Databricksでデータ分析を行う際に頻繁に使用するのが、今画面に表示している**ノートブック**です。AIアシスタントをはじめ、分析者の生産性を高めるための工夫が随所に施されています。

- [日本語設定](https://qiita.com/taka_yayoi/items/ff4127e0d632f5e02603)
- [ワークスペース](https://docs.databricks.com/ja/workspace/index.html)
- [ノートブック](https://docs.databricks.com/ja/notebooks/index.html)の作成

## 計算資源(コンピュート)

Databricksにおける計算資源は[コンピュート](https://docs.databricks.com/ja/compute/index.html)と呼ばれます。Databricksが全てを管理する**サーバレス**の計算資源も利用できます。

- コンピュート
- SQLウェアハウス

ここでは、画面の右上にある**接続**ボタンをおして、**サーバレスコンピュート**を選択します。

![](./compute1.png)

**接続済み**となれば計算資源を使用してプログラムを実行することができます。

![](./compute2.png)

プログラムを実行するにはセルの左上にある▶︎ボタンをクリックします。

In [0]:
print("Hello Databricks!")

## データの読み込み

Databricksでは上述のコンピュートを用いて、ファイルやテーブルにアクセスします。SQLやPython、Rなどを活用することができます。Databricksノートブックではこれらの[複数の言語を混在](https://docs.databricks.com/ja/notebooks/notebooks-code.html#mix-languages)させることができます。

![how_to_use2.png](./how_to_use2.png "how_to_use2.png")

- [ビジュアライゼーション/データプロファイル](https://docs.databricks.com/ja/visualizations/index.html)
- [AIアシスタント](https://docs.databricks.com/ja/notebooks/databricks-assistant-faq.html)

AIアシスタントを活用してロジックを組み立てることができます。アシスタントを呼び出すには下のセルにカーソルを合わせると右上に表示される![assistant.png](./assistant.png "assistant.png")アイコンをクリックします。そして、プロンプトボックスに以下のプロンプトを入力し、**生成**をクリックします。

![](prompt.png)

プロンプト: `samples.tpch.ordersの中身を1000行表示`

In [0]:
def calculator():
    print("電卓へようこそ")
    print("例: 2 + 3, 4 * 5, 10 / 2, 7 - 1")
    expr = input("計算式を入力してください: ")
    try:
        result = eval(expr)
        print(f"結果: {result}")
    except Exception as e:
        print("計算式が正しくありません。")

calculator()

## データの加工

AIアシスタントを活用してロジックを組み立てることができます。なお、アシスタントを活用する際には**具体的に**指示することが重要です。

プロンプト:`クエリーの処理に関する日本語コメントを追加して`

**参考資料**

- [Databricksアシスタントの新機能を試す](https://qiita.com/taka_yayoi/items/058f324960cecceef218)
- [Databricksアシスタントを用いたEDA\(探索的データ分析\)](https://qiita.com/taka_yayoi/items/07c49c2de588a101b719)
- [DatabricksアシスタントによるEDA\(探索的データ分析\) その2](https://qiita.com/taka_yayoi/items/29af5ef10aeba01ca391)
- [Databricksアシスタントによるデータエンジニアリング](https://qiita.com/taka_yayoi/items/b76ca7f8f11dce2be1c3)

In [0]:
%sql
-- samples.tpch.ordersテーブルから注文情報を抽出
SELECT
  o_orderkey,         -- 注文キー
  o_custkey,          -- 顧客キー
  o_orderstatus,      -- 注文ステータス
  o_totalprice,       -- 合計金額
  o_orderdate,        -- 注文日
  o_orderpriority     -- 注文優先度
FROM
  samples.tpch.orders
WHERE
  o_orderdate = "1998-07-01"    -- 1998年7月1日の注文のみ抽出
  AND o_totalprice >= 100000    -- 合計金額が100,000以上の注文のみ抽出

## データの書き込み

テーブルから別のテーブルを作成するといった作業をしていると、`このテーブルはどうやって作ったんだっけ？`となりがちです。そのような依存関係は、Databricksでは **リネージ(系統情報)** として自動で記録されます。

- [リネージ](https://docs.databricks.com/ja/data-governance/unity-catalog/data-lineage.html)

In [0]:
%sql
CREATE TABLE workspace.default.tpch_orders_199807 AS
SELECT
  o_orderkey,
  o_custkey,
  o_orderstatus,
  o_totalprice,
  o_orderdate,
  o_orderpriority
FROM
  samples.tpch.orders
WHERE
  o_orderdate = "1998-07-01"
  AND o_totalprice >= 100000

上で作成したテーブルから、さらに別のテーブルを作成します。

In [0]:
%sql
-- 注文優先度ごとの注文数を集計し、降順で表示するテーブルを作成
CREATE TABLE workspace.default.tpch_orders_199807_derived AS
SELECT
  COUNT(o_orderpriority) AS order_cnt,  -- 注文優先度ごとの注文数
  o_orderpriority                      -- 注文優先度
FROM
  workspace.default.tpch_orders_199807
GROUP BY
  o_orderpriority                      -- 注文優先度でグループ化
ORDER BY
  order_cnt DESC                       -- 注文数の降順で並び替え

[カタログエクスプローラ](/explore/data/workspace/default)でテーブルを確認しましょう。

また、[リネージ](/explore/data/workspace/default/tpch_orders_199807?activeTab=lineage)を確認してみましょう。**リネージグラフ**ボタンをクリックすることでより視覚的に関係性を確認することができます。

## クリーンアップ

In [0]:
%sql
DROP TABLE workspace.default.tpch_orders_199807_derived;
DROP TABLE workspace.default.tpch_orders_199807;

**参考資料**

- [Databricksドキュメント \| Databricks on AWS](https://docs.databricks.com/ja/index.html)
- [はじめてのDatabricks](https://qiita.com/taka_yayoi/items/8dc72d083edb879a5e5d)
- [Databricksチュートリアル](https://qiita.com/taka_yayoi/items/4603091dd325c77d577f)
- [Databricks記事のまとめページ\(その1\)](https://qiita.com/taka_yayoi/items/c6907e2b861cb1070f4d)
- [Databricks記事のまとめページ\(その2\)](https://qiita.com/taka_yayoi/items/68fc3d67880d2dcb32bb)